# Traductor en cadena con TranslateGemma 12B (rama `modelo-12b`)

Lo mismo que el notebook principal (re-anclaje con LaBSE, ida y vuelta, interfaz), pero con **TranslateGemma 12B**, el hermano mayor del 4B: misma arquitectura y formato, ~3 veces más parámetros. Sirve para ver si un modelo más grande traduce mejor las frases complejas, sobre todo en latín.

- Para caber en la **T4 gratuita** (15 GB) se carga **comprimido a 4 bits** (NF4): ~8.5 GB en vez de ~24.
- **No usa el adaptador de latín** (se entrenó para el 4B). No entrena nada.
- Vive en la rama `modelo-12b` del repo: el notebook principal y `main` no cambian.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)` y luego `Ejecutar todas`.

| Paso | Tiempo aprox. en T4 |
|---|---|
| Instalar y descargar el 12B (24 GB) + LaBSE | 10-15 min |
| Prueba con una frase | 1-3 min |
| Comparar con 10 frases complejas (2 configuraciones) | 30-45 min (estimado: el 12B es más lento) |
| Interfaz | al momento |

Al terminar la comparación se **descarga un zip** con los resultados. Usa las mismas 10 frases y la misma cadena que el notebook de reentrenamiento (v1 / v2), así que los números se pueden comparar directamente.

## 1. Verificar GPU

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No hay GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU.")

## 2. Obtener el código (rama `modelo-12b`) e instalar librerías

In [ ]:
REPO = "AndreaHortaHernandez/Traductormultilingue"
RAMA = "modelo-12b"

import os
if not os.path.isdir("/content/repo"):
    !git clone -b {RAMA} https://github.com/{REPO}.git /content/repo
else:
    !cd /content/repo && git fetch origin {RAMA} && git checkout {RAMA} && git pull origin {RAMA}
%cd /content/repo
!git log --oneline -1
!pip install -q -U transformers sentence-transformers sacrebleu peft bitsandbytes gradio
!pip uninstall -q -y torchao

## 3. Cargar TranslateGemma 12B (4 bits) y LaBSE

La primera vez descarga ~24 GB (se comprimen al cargarlos a la GPU).

In [ ]:
import time, torch, config
from modelos import Traductor, Verificador
inicio = time.time()
traductor = Traductor()
verificador = Verificador()
print(f"\n{config.NOMBRE_MODELO} listo en {(time.time() - inicio) / 60:.1f} min")
print(f"Memoria de GPU en uso: {torch.cuda.memory_allocated() / 1e9:.1f} GB de "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Probar una frase

In [ ]:
frase = "Quisiera comer algo diferente esta noche, pero no sé qué."  #@param {type:"string"}
cadena = "la ko hi la ko hi"  #@param {type:"string"}

from cadena import traducir_en_cadena, imprimir_pasos
inicio = time.time()
imprimir_pasos(frase, traducir_en_cadena(frase, cadena.split(), traductor, verificador))
print(f"\n({time.time() - inicio:.0f} s)")

## 5. Comparar con frases complejas

Las mismas 10 frases de `frases_complejas.txt` y la misma cadena que el notebook de reentrenamiento, para poner lado a lado **12B** contra **4B v1 / v2 / sin adaptador**. Desmarca la casilla para saltarla.

In [ ]:
COMPARAR = True  #@param {type:"boolean"}
limite = 10  #@param {type:"integer"}

if COMPARAR:
    import comparar
    comparacion = comparar.comparar(
        traductor, verificador, ruta_frases="frases_complejas.txt",
        limite=limite, variantes=comparar.VARIANTES_12B,
    )
    !cd /content/repo && zip -qr resultados_12b.zip salidas
    from google.colab import files
    files.download("/content/repo/resultados_12b.zip")

## 6. Interfaz web (con el 12B)

Igual que en el notebook principal; arriba dice qué modelo usa. Más lenta que con el 4B.

In [ ]:
import interfaz
demo = interfaz.lanzar(traductor, verificador)  # para cerrarla: demo.close()